# SKU Predictability Map

Исследовательский notebook для первого шага гибридной программы: понять, как ведут себя SKU-пары в зависимости от `R2`.

Что делает notebook:
- загружает итоговый гибридный CSV;
- строит SKU-level behavioral features по дневному спросу;
- бьёт SKU по диапазонам `R2`;
- показывает, чем плохие SKU отличаются от хороших;
- сохраняет промежуточные таблицы в `reports/hybrid_research/`.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / 'src').exists() and (ROOT.parent / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import display

from src.experiments_v2.benchmark_common import BAKERY_COL, DATE_COL, DEMAND_TARGET, PRODUCT_COL, load_daily_demand
from src.experiments_v2.monthly_benchmark_common import build_exp63_feature_frame
from src.experiments_v2.hybrid_sku_features import build_sku_stability_features
from src.experiments_v2.hybrid_sku_features import build_sku_stationarity_features

BENCHMARK_PATH = ROOT / 'src' / 'experiments_v2' / 'Full_benchmark_mounth_results' / 'src' / 'experiments_v2' / 'merged_best_by_sku.csv'
ARTIFACT_DIR = ROOT / 'reports' / 'hybrid_research'
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 160)
sns.set_theme(style='whitegrid')

print('ROOT:', ROOT)
print('Benchmark:', BENCHMARK_PATH)
print('Artifacts:', ARTIFACT_DIR)


In [ ]:
benchmark = pd.read_csv(BENCHMARK_PATH, encoding='utf-8-sig')
raw_df = load_daily_demand()
featured_df = build_exp63_feature_frame(raw_df)
stability_df = build_sku_stability_features(featured_df)
stationarity_df = build_sku_stationarity_features(featured_df)

sku_behavior = (
    featured_df.groupby([BAKERY_COL, PRODUCT_COL], sort=False)
    .agg(
        n_days=(DATE_COL, 'nunique'),
        mean_demand=(DEMAND_TARGET, 'mean'),
        median_demand=(DEMAND_TARGET, 'median'),
        std_demand=(DEMAND_TARGET, 'std'),
        share_zero=(DEMAND_TARGET, lambda s: float((s == 0).mean())),
        share_censored=('is_censored', 'mean'),
        lost_mean=('lost_qty', 'mean'),
        demand_trend_mean=('demand_trend', 'mean'),
        cv_7d_mean=('cv_7d', 'mean'),
    )
    .reset_index()
)
sku_behavior['cv_demand'] = sku_behavior['std_demand'].fillna(0) / (sku_behavior['mean_demand'].abs() + 1e-8)

merged = (
    benchmark.merge(sku_behavior, on=[BAKERY_COL, PRODUCT_COL], how='left', validate='one_to_one')
    .merge(stability_df, on=[BAKERY_COL, PRODUCT_COL], how='left', validate='one_to_one')
    .merge(stationarity_df, on=[BAKERY_COL, PRODUCT_COL], how='left', validate='one_to_one')
)
r2_bins = [-np.inf, 0, 0.2, 0.5, np.inf]
r2_labels = ['R2 < 0', '0..0.2', '0.2..0.5', '>= 0.5']
merged['r2_bin'] = pd.cut(merged['best_r2'], bins=r2_bins, labels=r2_labels, right=False)
merged['is_negative_r2'] = merged['best_r2'] < 0

print('Merged rows:', len(merged))
print('Negative R2 share:', round(float(merged['is_negative_r2'].mean()), 4))
display(merged.head(3))


In [ ]:
sku_r2_summary = merged[[
    BAKERY_COL,
    PRODUCT_COL,
    'best_model',
    'winner_source',
    'best_r2',
    'best_mae',
    'best_mse',
    'best_wmape',
    'r2_bin',
    'n_days',
    'mean_demand',
    'median_demand',
    'std_demand',
    'cv_demand',
    'share_zero',
    'share_censored',
    'lost_mean',
    'demand_trend_mean',
    'cv_7d_mean',
    'acf_lag1',
    'acf_lag7',
    'trend_abs_60',
    'mean_shift_norm_30_60',
    'variance_shift_norm_7_30',
    'seasonal_strength_7',
    'zero_run_max',
    'zero_run_ratio',
    'adf_pvalue',
    'adf_stat',
    'kpss_pvalue',
    'kpss_stat',
    'stationary_adf',
    'stationary_kpss',
]].sort_values('best_r2').reset_index(drop=True)

sku_r2_bins_profile = (
    merged.groupby('r2_bin', observed=False)
    .agg(
        n_sku=(PRODUCT_COL, 'size'),
        best_r2_mean=('best_r2', 'mean'),
        best_r2_median=('best_r2', 'median'),
        best_mae_mean=('best_mae', 'mean'),
        best_wmape_mean=('best_wmape', 'mean'),
        mean_demand_mean=('mean_demand', 'mean'),
        median_demand_mean=('median_demand', 'mean'),
        share_zero_mean=('share_zero', 'mean'),
        cv_demand_mean=('cv_demand', 'mean'),
        share_censored_mean=('share_censored', 'mean'),
        acf_lag1_mean=('acf_lag1', 'mean'),
        acf_lag7_mean=('acf_lag7', 'mean'),
        trend_abs_60_mean=('trend_abs_60', 'mean'),
        mean_shift_norm_30_60_mean=('mean_shift_norm_30_60', 'mean'),
        variance_shift_norm_7_30_mean=('variance_shift_norm_7_30', 'mean'),
        seasonal_strength_7_mean=('seasonal_strength_7', 'mean'),
        zero_run_max_mean=('zero_run_max', 'mean'),
        zero_run_ratio_mean=('zero_run_ratio', 'mean'),
        adf_pvalue_mean=('adf_pvalue', 'mean'),
        adf_stat_mean=('adf_stat', 'mean'),
        kpss_pvalue_mean=('kpss_pvalue', 'mean'),
        kpss_stat_mean=('kpss_stat', 'mean'),
        stationary_adf_share=('stationary_adf', 'mean'),
        stationary_kpss_share=('stationary_kpss', 'mean'),
    )
    .reset_index()
)

stability_cols = [
    'acf_lag1',
    'acf_lag7',
    'trend_abs_60',
    'mean_shift_norm_30_60',
    'variance_shift_norm_7_30',
    'seasonal_strength_7',
    'zero_run_max',
    'zero_run_ratio',
    'adf_pvalue',
    'adf_stat',
    'kpss_pvalue',
    'kpss_stat',
    'stationary_adf',
    'stationary_kpss',
]
stability_corr = (
    merged[['best_r2'] + stability_cols]
    .corr(numeric_only=True)['best_r2']
    .drop(labels=['best_r2'])
    .sort_values(ascending=False)
    .to_frame(name='corr_with_best_r2')
)
stability_by_bin = (
    merged.groupby('r2_bin', observed=False)[stability_cols]
    .mean()
    .reset_index()
)
stationarity_summary = pd.DataFrame({
    'metric': ['adf_pvalue', 'kpss_pvalue', 'stationary_adf_share', 'stationary_kpss_share'],
    'overall': [
        float(merged['adf_pvalue'].mean()),
        float(merged['kpss_pvalue'].mean()),
        float(merged['stationary_adf'].mean()),
        float(merged['stationary_kpss'].mean()),
    ],
})
stationarity_by_bin = (
    merged.groupby('r2_bin', observed=False)
    .agg(
        adf_pvalue_mean=('adf_pvalue', 'mean'),
        kpss_pvalue_mean=('kpss_pvalue', 'mean'),
        stationary_adf_share=('stationary_adf', 'mean'),
        stationary_kpss_share=('stationary_kpss', 'mean'),
    )
    .reset_index()
)

model_win_by_r2_bin = (
    merged.groupby(['r2_bin', 'best_model'], observed=False)
    .size()
    .reset_index(name='win_count')
)
model_win_pivot = model_win_by_r2_bin.pivot(index='best_model', columns='r2_bin', values='win_count').fillna(0).astype(int)

display(sku_r2_bins_profile)
display(stability_corr)
display(stability_by_bin)
display(stationarity_summary)
display(stationarity_by_bin)
display(model_win_pivot)

sku_r2_summary.to_csv(ARTIFACT_DIR / 'sku_r2_summary.csv', index=False, encoding='utf-8-sig')
sku_r2_bins_profile.to_csv(ARTIFACT_DIR / 'sku_r2_bins_profile.csv', index=False, encoding='utf-8-sig')
stability_corr.to_csv(ARTIFACT_DIR / 'stability_corr.csv', encoding='utf-8-sig')
stability_by_bin.to_csv(ARTIFACT_DIR / 'stability_by_bin.csv', index=False, encoding='utf-8-sig')
stationarity_summary.to_csv(ARTIFACT_DIR / 'stationarity_summary.csv', index=False, encoding='utf-8-sig')
stationarity_by_bin.to_csv(ARTIFACT_DIR / 'stationarity_by_bin.csv', index=False, encoding='utf-8-sig')
model_win_by_r2_bin.to_csv(ARTIFACT_DIR / 'model_win_by_r2_bin.csv', index=False, encoding='utf-8-sig')
print('Saved summary tables to', ARTIFACT_DIR)


In [ ]:
# Failure profile: why SKU are hard to predict
# Priority order: data-poor -> low-signal -> censored -> regime shift -> regular seasonal -> mixed
failure_profile = merged.copy()
failure_profile['is_bad_r2'] = failure_profile['best_r2'] < 0
failure_profile['is_data_poor'] = failure_profile['n_days'] <= 54
failure_profile['is_low_signal'] = (
    (failure_profile['mean_demand'] <= 2.874891)
    & (failure_profile['cv_demand'] >= 0.608539)
)
failure_profile['is_censored_heavy'] = (
    (failure_profile['share_censored'] >= 0.649029)
    | (failure_profile['lost_mean'] >= 1.620777)
)
failure_profile['is_regime_shift'] = (
    (failure_profile['mean_shift_norm_30_60'] >= 0.165680)
    | (failure_profile['variance_shift_norm_7_30'] >= 0.343036)
    | (failure_profile['trend_abs_60'] >= 0.005659)
)
failure_profile['is_regular_seasonal'] = (
    (failure_profile['seasonal_strength_7'] >= 0.402863)
    & (failure_profile['acf_lag7'] >= 0.154432)
    & (failure_profile['mean_demand'] >= 9.315062)
    & (failure_profile['cv_demand'] <= 0.608539)
)

reason_conditions = [
    failure_profile['is_data_poor'],
    failure_profile['is_low_signal'],
    failure_profile['is_censored_heavy'],
    failure_profile['is_regime_shift'],
    failure_profile['is_regular_seasonal'],
]
reason_choices = [
    'data_poor',
    'low_signal_sparse',
    'censored_heavy',
    'regime_shift',
    'regular_seasonal',
]
failure_profile['primary_reason'] = np.select(
    reason_conditions,
    reason_choices,
    default='mixed_or_other',
)

# Keep all matching flags for diagnostics, but make the primary label exclusive.
failure_profile['primary_reason'] = pd.Categorical(
    failure_profile['primary_reason'],
    categories=['data_poor', 'low_signal_sparse', 'censored_heavy', 'regime_shift', 'regular_seasonal', 'mixed_or_other'],
    ordered=True,
)

failure_reason_summary = (
    failure_profile.groupby('primary_reason', observed=False)
    .agg(
        n_sku=(PRODUCT_COL, 'size'),
        bad_r2_share=('is_bad_r2', 'mean'),
        avg_r2=('best_r2', 'mean'),
        median_r2=('best_r2', 'median'),
        avg_mean_demand=('mean_demand', 'mean'),
        avg_cv_demand=('cv_demand', 'mean'),
        avg_share_censored=('share_censored', 'mean'),
        avg_lost_mean=('lost_mean', 'mean'),
    )
    .sort_values('n_sku', ascending=False)
    .reset_index()
)

bad_reason_summary = (
    failure_profile[failure_profile['is_bad_r2']]
    .groupby('primary_reason', observed=False)
    .agg(
        n_sku=(PRODUCT_COL, 'size'),
        avg_r2=('best_r2', 'mean'),
        median_r2=('best_r2', 'median'),
        avg_mean_demand=('mean_demand', 'mean'),
        avg_cv_demand=('cv_demand', 'mean'),
        avg_share_censored=('share_censored', 'mean'),
    )
    .sort_values('n_sku', ascending=False)
    .reset_index()
)

failure_flags = failure_profile[[
    BAKERY_COL,
    PRODUCT_COL,
    'best_r2',
    'primary_reason',
    'is_bad_r2',
    'is_data_poor',
    'is_low_signal',
    'is_censored_heavy',
    'is_regime_shift',
    'is_regular_seasonal',
    'mean_demand',
    'cv_demand',
    'share_censored',
    'lost_mean',
    'acf_lag7',
    'seasonal_strength_7',
    'mean_shift_norm_30_60',
    'variance_shift_norm_7_30',
    'trend_abs_60',
    'n_days',
]].copy()

display(failure_reason_summary)
display(bad_reason_summary)
display(failure_flags.head(20))

failure_reason_summary.to_csv(ARTIFACT_DIR / 'failure_reason_summary.csv', index=False, encoding='utf-8-sig')
bad_reason_summary.to_csv(ARTIFACT_DIR / 'bad_reason_summary.csv', index=False, encoding='utf-8-sig')
failure_flags.to_csv(ARTIFACT_DIR / 'failure_flags.csv', index=False, encoding='utf-8-sig')
print('Saved failure profile tables to', ARTIFACT_DIR)


In [ ]:
r2_low = merged['best_r2'].quantile(0.01)
r2_high = merged['best_r2'].quantile(0.99)
r2_zoom = merged['best_r2'].clip(lower=r2_low, upper=r2_high)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

sns.histplot(r2_zoom, bins=60, kde=True, ax=axes[0, 0], color='#264653')
axes[0, 0].axvline(0, color='crimson', linestyle='--', linewidth=1)
axes[0, 0].set_title('Распределение best_r2 (zoom 1%-99%)')
axes[0, 0].set_xlabel('best_r2')
axes[0, 0].set_xlim(r2_low, r2_high)

sns.boxplot(data=merged, x='r2_bin', y='mean_demand', ax=axes[0, 1], color='#8ecae6')
axes[0, 1].set_title('Средний спрос по бинам R2')
axes[0, 1].set_xlabel('Бин R2')

sns.boxplot(data=merged, x='r2_bin', y='share_zero', ax=axes[1, 0], color='#ffb703')
axes[1, 0].set_title('Доля нулей по бинам R2')
axes[1, 0].set_xlabel('Бин R2')

sns.boxplot(data=merged, x='r2_bin', y='cv_demand', ax=axes[1, 1], color='#90be6d')
axes[1, 1].set_title('Коэффициент вариации по бинам R2')
axes[1, 1].set_xlabel('Бин R2')

plt.tight_layout()
plt.show()

fig = plt.figure(figsize=(16, 6))
sample = merged.sample(n=min(len(merged), 5000), random_state=42)

ax1 = plt.subplot(1, 2, 1)
sns.scatterplot(data=sample, x='mean_demand', y='best_r2', hue='r2_bin', palette='viridis', s=24, alpha=0.55, ax=ax1)
ax1.set_title('mean_demand vs best_r2')
ax1.set_xlabel('mean_demand')
ax1.set_ylabel('best_r2')

ax2 = plt.subplot(1, 2, 2)
sns.scatterplot(data=sample, x='cv_demand', y='best_r2', hue='r2_bin', palette='viridis', s=24, alpha=0.55, ax=ax2, legend=False)
ax2.set_title('cv_demand vs best_r2')
ax2.set_xlabel('cv_demand')
ax2.set_ylabel('best_r2')

plt.tight_layout()
plt.show()


## Победы моделей по бинам R2

Heatmap показывает, какая модель чаще всего выигрывает внутри каждого диапазона `R2`.


In [ ]:
fig = plt.figure(figsize=(10, 6))
heatmap_data = model_win_pivot.loc[model_win_pivot.sum(axis=1).sort_values(ascending=False).index]
sns.heatmap(heatmap_data, annot=True, fmt='d', cmap='Blues')
plt.title('Победы моделей по бинам R2')
plt.xlabel('Бин R2')
plt.ylabel('Модель')
plt.tight_layout()
plt.show()


## Что смотреть после запуска

- сколько SKU попадает в `R2 < 0`;
- отличаются ли эти SKU по среднему спросу, доле нулей, CV и цензуре;
- какие модели чаще выигрывают в каждом бине;
- стоит ли выделять отдельный fallback-класс для устойчиво слабых SKU.

Следующий логичный шаг после этого ноутбука - формализовать правила сегментации SKU и проверить, помогает ли исключение или downweighting плохих рядов при обучении глобальной модели.
